In [2]:

import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
# Set the parent directory as the current directory
os.chdir(parent_dir)

In [11]:
import json
import pandas as pd
import argparse
from pathlib import Path

def extract_flagged_entities(json_file):
    """Extract flagged entities with context from supervisor output JSON file."""
    
    # Load the JSON file
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Get the flagged entities from the summary
    flagged_entities = data.get('summary', {}).get('flagged_entities', [])
    print(f"Found {len(flagged_entities)} flagged entities")
    
    # Prepare detailed data for review
    detailed_entities = []
    
    # Process each flagged entity
    for entity in flagged_entities:
        entity_info = {
            'entity': entity.get('entity', ''),
            'document_id': entity.get('document_id', ''),
            'orpha_code': entity.get('orpha_code', ''),
            'category': entity.get('category', ''),
            'explanation': entity.get('explanation', '')
        }
        
        # Get detailed information from the results section
        category = entity.get('category', '')
        doc_id = entity.get('document_id', '')
        
        if doc_id and category and category in data.get('results', {}):
            # Find the detailed entity data
            for result in data['results'][category]:
                if (result.get('document_id') == doc_id and 
                    result.get('entity') == entity_info['entity']):
                    # Add detailed info
                    entity_info['context'] = result.get('context', '')
                    entity_info['is_rare_disease'] = result.get('is_rare_disease', False)
                    
                    # Add top candidate
                    candidates = result.get('orpha_candidates', [])
                    if candidates:
                        top_candidate = candidates[0]
                        entity_info['top_candidate_name'] = top_candidate.get('name', '')
                        entity_info['top_candidate_id'] = top_candidate.get('id', '')
                        entity_info['top_candidate_similarity'] = top_candidate.get('similarity', 0.0)
                    
                    break
        
        detailed_entities.append(entity_info)
    
    return detailed_entities

def main():
    # parser = argparse.ArgumentParser(description="Extract flagged entities for review")
    # parser.add_argument("json_file", help="Path to supervisor output JSON file")
    # parser.add_argument("--output", help="Output CSV file for review (optional)")
    # parser.add_argument("--category", choices=["false_positives", "false_negatives", "true_positives"],
    #                   help="Filter by category (optional)")
    
    # args = parser.parse_args()
    
    # Extract entities
    json_file = "data/results/supervisor/multistage_no_min.json"
    category = "false_positives"
    entities = extract_flagged_entities(json_file)
    print(entities[0])
    # Filter by category if requested
    if category:
        entities = [e for e in entities if e.get('category') == category]
        print(f"Filtered to {len(entities)} {category}")
    
    # Create dataframe
    df = pd.DataFrame(entities)
    
    
if __name__ == "__main__":
    main()

Found 43 flagged entities
{'entity': 'hemochromatosis', 'document_id': '11604', 'orpha_code': '220489', 'category': 'false_negatives', 'explanation': 'low_orpha_similarity [FLAGGED: Entity verified as not a rare disease despite being categorized as false negative]', 'context': ' w/ h/o SMA thrombosis s/p partial small bowel\nresection, possible protein C deficiency, asthma and\nhemochromatosis who presents with a 4 day history of persistent\nL leg pain and swelling. The patient has also exper', 'is_rare_disease': False, 'top_candidate_name': 'hemochromatosis due to defect in ferroportin', 'top_candidate_id': 'Orpha:139491', 'top_candidate_similarity': 0.7696985373209294}
Filtered to 28 false_positives


# Create a new set of annotations frome existing ones and human ones.

In [19]:
import json
from typing import Dict, List, Any
from datetime import datetime
import os

def detailed_correction_diagnostic(
    existing_annotations: Dict[str, Any], 
    corrections: Dict[str, Any]
) -> None:
    """
    Provide detailed diagnostic information about the correction process.
    
    Args:
        existing_annotations (Dict): Original annotations dictionary
        corrections (Dict): Corrections dictionary
    """
    print("\n===== CORRECTION DIAGNOSTIC =====")
    
    # Print basic information about corrections
    correction_list = corrections.get('corrected_annotations', [])
    print(f"Total corrections found: {len(correction_list)}")
    
    # Print detailed information about each correction
    for i, correction in enumerate(correction_list, 1):
        print(f"\nCorrection {i}:")
        print("Correction Details:")
        for key, value in correction.items():
            print(f"  {key}: {value}")
        
        # Try to find corresponding document
        document_id = str(correction.get('document_id', ''))
        if document_id in existing_annotations:
            print(f"  Document {document_id} exists in annotations")
            
            # Check existing annotations for this document
            doc_annotations = existing_annotations[document_id].get('annotations', [])
            entity = correction.get('entity', '')
            
            # Look for matching mentions
            matching_mentions = [
                ann for ann in doc_annotations 
                if ann.get('mention', '').lower() == entity.lower()
            ]
            
            print(f"  Matching mentions in document: {len(matching_mentions)}")
            if matching_mentions:
                for match in matching_mentions:
                    print("  Existing annotation details:")
                    print(json.dumps(match, indent=2))
        else:
            print(f"  Document {document_id} NOT FOUND in annotations")

def update_annotations_with_corrections(
    existing_annotations: Dict[str, Any], 
    corrections: Dict[str, Any], 
    output_file: str = None,
    debug: bool = True
) -> Dict[str, Any]:
    """
    Update existing annotations with corrections from the review process.
    
    Args:
        existing_annotations (Dict): Original annotations dictionary
        corrections (Dict): Corrections from the review process
        output_file (str, optional): Path to save the updated annotations
        debug (bool, optional): Enable debug output
    
    Returns:
        Dict: Updated annotations dictionary with enriched rare disease annotations
    """
    # Print diagnostic information if debug is on
    if debug:
        detailed_correction_diagnostic(existing_annotations, corrections)
    
    # Validate input
    if not isinstance(existing_annotations, dict):
        raise ValueError("Existing annotations must be a dictionary")
    
    if not isinstance(corrections, dict) or 'corrected_annotations' not in corrections:
        raise ValueError("Corrections must be a dictionary with 'corrected_annotations' key")
    
    # Create a deep copy of existing annotations to modify
    import copy
    updated_annotations = copy.deepcopy(existing_annotations)
    
    # Tracking statistics
    stats = {
        'total_corrections': 0,
        'added_annotations': 0,
        'updated_annotations': 0,
        'skipped_corrections': 0,
        'documents_modified': []
    }
    
    # Process each correction
    for correction in corrections.get('corrected_annotations', []):
        # Increment total corrections
        stats['total_corrections'] += 1
        
        # Extract key information
        document_id = str(correction.get('document_id', ''))
        entity = correction.get('entity', '')
        orpha_code = correction.get('orpha_code', '')
        is_rare_disease = correction.get('is_rare_disease', False)
        
        if debug:
            print(f"\nProcessing Correction:")
            print(f"  Document ID: {document_id}")
            print(f"  Entity: {entity}")
            print(f"  ORPHA Code: {orpha_code}")
            print(f"  Is Rare Disease: {is_rare_disease}")
        
        # Skip if critical information is missing
        if not document_id or not entity or not orpha_code:
            if debug:
                print("  Skipping: Missing critical information")
            stats['skipped_corrections'] += 1
            continue
        
        # Skip if not a confirmed rare disease
        if not is_rare_disease:
            if debug:
                print("  Skipping: Not a confirmed rare disease")
            stats['skipped_corrections'] += 1
            continue
        
        # Check if the document exists in existing annotations
        if document_id not in updated_annotations:
            if debug:
                print(f"  Document {document_id} not found in annotations")
            stats['skipped_corrections'] += 1
            continue
        
        # Ensure annotations list exists
        if 'annotations' not in updated_annotations[document_id]:
            updated_annotations[document_id]['annotations'] = []
        
        # Try to find and update existing annotation
        annotations = updated_annotations[document_id]['annotations']
        found_match = False
        
        for annotation in annotations:
            # Match by mention, case-insensitive
            if annotation.get('mention', '').lower() == entity.lower():
                # Update the annotation with ORPHA code
                annotation['ordo_with_desc'] = f"ORPHA:{orpha_code} {entity}"
                found_match = True
                stats['updated_annotations'] += 1
                
                if debug:
                    print(f"  Updated existing annotation for {entity}")
                
                # Track document modification
                if document_id not in stats['documents_modified']:
                    stats['documents_modified'].append(document_id)
                break
        
        # If no match found, add new annotation
        if not found_match:
            new_annotation = {
                'mention': entity,
                'umls_with_desc': '',
                'ordo_with_desc': f"ORPHA:{orpha_code} {entity}",
                'gold_text_to_umls_label': 0,
                'gold_text_to_ordo_label': 1,
                'document_structure': '',
                'semehr_label': 0,
                'correction_source': 'iterative_verification'
            }
            
            annotations.append(new_annotation)
            stats['added_annotations'] += 1
            
            if debug:
                print(f"  Added new annotation for {entity}")
            
            # Track document modification
            if document_id not in stats['documents_modified']:
                stats['documents_modified'].append(document_id)
    
    # Prepare output metadata
    output_metadata = {
        'update_timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'stats': stats
    }
    
    # Create final output dictionary
    final_output = {
        'metadata': output_metadata,
        'documents': updated_annotations
    }
    
    # Save to output file if specified
    if output_file:
        os.makedirs(os.path.dirname(os.path.abspath(output_file)), exist_ok=True)
        with open(output_file, 'w') as f:
            json.dump(final_output, f, indent=2)
    
    # Print summary
    if debug:
        print("\n=== Correction Processing Summary ===")
        for key, value in stats.items():
            print(f"{key}: {value}")
    
    return final_output

def process_annotations_correction(
    existing_annotations_path: str, 
    corrections_path: str, 
    output_path: str = None,
    debug: bool = True
) -> Dict[str, Any]:
    """
    Convenience function to load and process annotations corrections.
    
    Args:
        existing_annotations_path (str): Path to existing annotations JSON file
        corrections_path (str): Path to corrections JSON file
        output_path (str, optional): Path to save updated annotations
        debug (bool, optional): Enable debug output
    
    Returns:
        Dict: Updated annotations dictionary
    """
    # Load existing annotations
    try:
        with open(existing_annotations_path, 'r') as f:
            existing_annotations = json.load(f)
    except Exception as e:
        print(f"Error loading existing annotations: {e}")
        raise
    
    # Load corrections
    try:
        with open(corrections_path, 'r') as f:
            corrections = json.load(f)
    except Exception as e:
        print(f"Error loading corrections: {e}")
        raise
    
    # Update and save annotations
    return update_annotations_with_corrections(
        existing_annotations, 
        corrections, 
        output_path, 
        debug=debug
    )

# Convenience function for manual debugging
def print_annotations_differences(
    original_path: str, 
    corrected_path: str
) -> None:
    """
    Print differences between original and corrected annotations.
    
    Args:
        original_path (str): Path to original annotations file
        corrected_path (str): Path to corrected annotations file
    """
    # Load original and corrected annotations
    with open(original_path, 'r') as f:
        original_annos = json.load(f)
    
    with open(corrected_path, 'r') as f:
        corrected_annos = json.load(f)
    
    print("\n===== ANNOTATION DIFFERENCES =====")
    
    # Compare document IDs with annotations
    for doc_id, doc_data in corrected_annos['documents'].items():
        # Check if document exists in original and has annotations
        if doc_id in original_annos and 'annotations' in doc_data:
            print(f"\nDocument {doc_id}:")
            
            # Original annotations
            orig_annos = original_annos.get(doc_id, {}).get('annotations', [])
            corr_annos = doc_data['annotations']
            
            # Compare annotations
            for corr_ann in corr_annos:
                # Check if this is a new or modified annotation
                matching_orig = [
                    orig for orig in orig_annos 
                    if orig.get('mention', '').lower() == corr_ann.get('mention', '').lower()
                ]
                
                if not matching_orig:
                    print("  New Annotation:")
                    print(json.dumps(corr_ann, indent=2))
                else:
                    # Check for differences
                    for orig in matching_orig:
                        # Compare specific fields that might have changed
                        if orig.get('ordo_with_desc') != corr_ann.get('ordo_with_desc'):
                            print("  Updated Annotation:")
                            print("  Original:")
                            print(json.dumps(orig, indent=2))
                            print("  Corrected:")
                            print(json.dumps(corr_ann, indent=2))

# Example usage (commented out)
updated_annotations = process_annotations_correction(
    "data/dataset/filtered_rd_annos_updated_adam.json", 
    "data/dataset/adam_corrections.json", 
    "data/dataset/rd_annos_adam_corrected_v1.json",
    debug=True
)


===== CORRECTION DIAGNOSTIC =====
Total corrections found: 43

Correction 1:
Correction Details:
  entity: hemochromatosis
  document_id: 11604
  orpha_code: 220489
  category: false_negatives
  is_rare_disease: False
  decision_timestamp: 2025-04-24T15:26:58.287Z
  Document 11604 exists in annotations
  Matching mentions in document: 6
  Existing annotation details:
{
  "mention": "hemochromatosis",
  "umls_with_desc": "C0018995 Hemochromatose",
  "ordo_with_desc": "Orphanet_220489  Rare hereditary hemochromatosis",
  "gold_text_to_umls_label": 1,
  "gold_text_to_ordo_label": 0,
  "document_structure": "Discharge_Diagnosis",
  "semehr_label": 1
}
  Existing annotation details:
{
  "mention": "hemochromatosis",
  "umls_with_desc": "C0018995 Hemochromatose",
  "ordo_with_desc": "Orphanet_220489  Rare hereditary hemochromatosis",
  "gold_text_to_umls_label": 1,
  "gold_text_to_ordo_label": 0,
  "document_structure": "History_of_Past_Illness",
  "semehr_label": 1
}
  Existing annotation 